In [60]:
import re
import string
import nltk

nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /usr/share/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /usr/share/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [61]:
with open("/kaggle/input/datasets/prabhavsinghal/next-word-data/next_word_predictor.txt", "r", encoding="utf-8") as file:
    document = file.read()

print(document[:1000])

The sun was shining brightly in the clear blue sky, and a gentle breeze rustled the leaves of the tall trees. People were out enjoying the beautiful weather, some sitting in the park, others taking a leisurely stroll along the riverbank. Children were playing games, and laughter filled the air.

As the day turned into evening, the temperature started to drop, and the sky transformed into a canvas of vibrant colors. Families gathered for picnics, and the smell of barbecues wafted through the air. It was a perfect day for a picnic by the lake.

In the distance, you could hear the sound of live music coming from a local band, and people began to gather around the stage to enjoy the performance. The atmosphere was electric, and the music had everyone swaying to the beat.

As the stars began to twinkle in the night sky, the crowd grew even larger, and the festivities continued well into the night. It was a day filled with joy, laughter, and memories that would last a lifetime.


The ancient

In [62]:
# lowercase
document = document.lower()

In [63]:
# removing urls
document = re.sub(r'https?://\S+|www\.\S+', '', document)

In [64]:
sentences = nltk.sent_tokenize(document)

print("Number of sentences:", len(sentences))

print()

print(sentences[:5])

Number of sentences: 2528

['the sun was shining brightly in the clear blue sky, and a gentle breeze rustled the leaves of the tall trees.', 'people were out enjoying the beautiful weather, some sitting in the park, others taking a leisurely stroll along the riverbank.', 'children were playing games, and laughter filled the air.', 'as the day turned into evening, the temperature started to drop, and the sky transformed into a canvas of vibrant colors.', 'families gathered for picnics, and the smell of barbecues wafted through the air.']


In [65]:
# removing punctuations
input_sentences = []

for sentence in sentences:

    sentence = sentence.translate(

        str.maketrans('', '', string.punctuation)

    )

    input_sentences.append(sentence)

print(input_sentences[:5])

['the sun was shining brightly in the clear blue sky and a gentle breeze rustled the leaves of the tall trees', 'people were out enjoying the beautiful weather some sitting in the park others taking a leisurely stroll along the riverbank', 'children were playing games and laughter filled the air', 'as the day turned into evening the temperature started to drop and the sky transformed into a canvas of vibrant colors', 'families gathered for picnics and the smell of barbecues wafted through the air']


In [66]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from collections import Counter
from torch.utils.data import Dataset, DataLoader
from nltk.tokenize import word_tokenize
import nltk

In [67]:
tokens = word_tokenize(document)

In [68]:
vocab = {'<unk>':0}

for token in Counter(tokens).keys():
  if token not in vocab:
    vocab[token] = len(vocab)

list(vocab.items())[:10]

[('<unk>', 0),
 ('the', 1),
 ('sun', 2),
 ('was', 3),
 ('shining', 4),
 ('brightly', 5),
 ('in', 6),
 ('clear', 7),
 ('blue', 8),
 ('sky', 9)]

In [69]:
len(vocab)

5058

In [70]:
input_sentences[:10]

['the sun was shining brightly in the clear blue sky and a gentle breeze rustled the leaves of the tall trees',
 'people were out enjoying the beautiful weather some sitting in the park others taking a leisurely stroll along the riverbank',
 'children were playing games and laughter filled the air',
 'as the day turned into evening the temperature started to drop and the sky transformed into a canvas of vibrant colors',
 'families gathered for picnics and the smell of barbecues wafted through the air',
 'it was a perfect day for a picnic by the lake',
 'in the distance you could hear the sound of live music coming from a local band and people began to gather around the stage to enjoy the performance',
 'the atmosphere was electric and the music had everyone swaying to the beat',
 'as the stars began to twinkle in the night sky the crowd grew even larger and the festivities continued well into the night',
 'it was a day filled with joy laughter and memories that would last a lifetime']

In [71]:
def text_to_indices(sentence,vocab):
    numerical_sentence = []
    for token in sentence:
        if token in vocab:
            numerical_sentence.append(vocab[token])
        else:
            numerical_sentence.append(vocab['<unk>'])
    return numerical_sentence

In [72]:
input_numerical_sentences = []
for sentence in input_sentences:
    input_numerical_sentences.append(text_to_indices(word_tokenize(sentence.lower()),vocab))

In [73]:
len(input_numerical_sentences)

2528

In [74]:
training_sequence = []
for sentence in input_numerical_sentences:

  for i in range(1, len(sentence)):
    training_sequence.append(sentence[:i+1])

In [75]:
len(training_sequence)

25180

In [78]:
training_sequence[:100]

[[1, 2],
 [1, 2, 3],
 [1, 2, 3, 4],
 [1, 2, 3, 4, 5],
 [1, 2, 3, 4, 5, 6],
 [1, 2, 3, 4, 5, 6, 1],
 [1, 2, 3, 4, 5, 6, 1, 7],
 [1, 2, 3, 4, 5, 6, 1, 7, 8],
 [1, 2, 3, 4, 5, 6, 1, 7, 8, 9],
 [1, 2, 3, 4, 5, 6, 1, 7, 8, 9, 11],
 [1, 2, 3, 4, 5, 6, 1, 7, 8, 9, 11, 12],
 [1, 2, 3, 4, 5, 6, 1, 7, 8, 9, 11, 12, 13],
 [1, 2, 3, 4, 5, 6, 1, 7, 8, 9, 11, 12, 13, 14],
 [1, 2, 3, 4, 5, 6, 1, 7, 8, 9, 11, 12, 13, 14, 15],
 [1, 2, 3, 4, 5, 6, 1, 7, 8, 9, 11, 12, 13, 14, 15, 1],
 [1, 2, 3, 4, 5, 6, 1, 7, 8, 9, 11, 12, 13, 14, 15, 1, 16],
 [1, 2, 3, 4, 5, 6, 1, 7, 8, 9, 11, 12, 13, 14, 15, 1, 16, 17],
 [1, 2, 3, 4, 5, 6, 1, 7, 8, 9, 11, 12, 13, 14, 15, 1, 16, 17, 1],
 [1, 2, 3, 4, 5, 6, 1, 7, 8, 9, 11, 12, 13, 14, 15, 1, 16, 17, 1, 18],
 [1, 2, 3, 4, 5, 6, 1, 7, 8, 9, 11, 12, 13, 14, 15, 1, 16, 17, 1, 18, 19],
 [21, 22],
 [21, 22, 23],
 [21, 22, 23, 24],
 [21, 22, 23, 24, 1],
 [21, 22, 23, 24, 1, 25],
 [21, 22, 23, 24, 1, 25, 26],
 [21, 22, 23, 24, 1, 25, 26, 27],
 [21, 22, 23, 24, 1, 25, 26, 27, 28]

In [79]:
len_list = []

for sequence in training_sequence:
  len_list.append(len(sequence))

max(len_list)

70

In [80]:
padded_training_sequence = []
for sequence in training_sequence:

  padded_training_sequence.append([0]*(max(len_list) - len(sequence)) + sequence)

In [81]:
len(padded_training_sequence[10])

70

In [82]:
padded_training_sequence = torch.tensor(padded_training_sequence, dtype=torch.long)

In [83]:
padded_training_sequence

tensor([[   0,    0,    0,  ...,    0,    1,    2],
        [   0,    0,    0,  ...,    1,    2,    3],
        [   0,    0,    0,  ...,    2,    3,    4],
        ...,
        [   0,    0,    0,  ..., 4773, 2297,  111],
        [   0,    0,    0,  ..., 2297,  111,  225],
        [   0,    0,    0,  ...,  111,  225, 4774]])

In [84]:
X = padded_training_sequence[:, :-1]
y = padded_training_sequence[:,-1]

In [85]:
X

tensor([[   0,    0,    0,  ...,    0,    0,    1],
        [   0,    0,    0,  ...,    0,    1,    2],
        [   0,    0,    0,  ...,    1,    2,    3],
        ...,
        [   0,    0,    0,  ...,  154, 4773, 2297],
        [   0,    0,    0,  ..., 4773, 2297,  111],
        [   0,    0,    0,  ..., 2297,  111,  225]])

In [86]:
y

tensor([   2,    3,    4,  ...,  111,  225, 4774])

In [87]:
class CustomDataset(Dataset):

  def __init__(self, X, y):
    self.X = X
    self.y = y

  def __len__(self):
    return self.X.shape[0]

  def __getitem__(self, idx):
    return self.X[idx], self.y[idx]

In [88]:
dataset = CustomDataset(X,y)

In [89]:
len(dataset)

25180

In [90]:
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

In [91]:
	
class LSTMModel(nn.Module):

  def __init__(self, vocab_size):
    super().__init__()
    self.embedding = nn.Embedding(vocab_size, 100)
    self.lstm = nn.LSTM(100, 150, batch_first=True)
    self.fc = nn.Linear(150, vocab_size)

  def forward(self, x):
    embedded = self.embedding(x)
    intermediate_hidden_states, (final_hidden_state, final_cell_state) = self.lstm(embedded)
    output = self.fc(final_hidden_state.squeeze(0))
    return output

In [92]:
model = LSTMModel(len(vocab))

In [93]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [94]:
model.to(device)

LSTMModel(
  (embedding): Embedding(5058, 100)
  (lstm): LSTM(100, 150, batch_first=True)
  (fc): Linear(in_features=150, out_features=5058, bias=True)
)

In [95]:
epochs = 50
learning_rate = 0.001

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

In [98]:
# training loop

for epoch in range(epochs):
  total_loss = 0

  for batch_x, batch_y in dataloader:

    batch_x, batch_y = batch_x.to(device), batch_y.to(device)

    optimizer.zero_grad()

    output = model(batch_x)

    loss = criterion(output, batch_y)

    loss.backward()

    optimizer.step()

    total_loss = total_loss + loss.item()
    avg_loss = total_loss/ len(dataloader)

  print(f"Epoch: {epoch + 1}, Loss: {avg_loss:.4f}")

Epoch: 1, Loss: 1.0630
Epoch: 2, Loss: 0.9472
Epoch: 3, Loss: 0.8560
Epoch: 4, Loss: 0.7807
Epoch: 5, Loss: 0.7217
Epoch: 6, Loss: 0.6725
Epoch: 7, Loss: 0.6305
Epoch: 8, Loss: 0.5979
Epoch: 9, Loss: 0.5695
Epoch: 10, Loss: 0.5457
Epoch: 11, Loss: 0.5295
Epoch: 12, Loss: 0.5175
Epoch: 13, Loss: 0.4991
Epoch: 14, Loss: 0.4891
Epoch: 15, Loss: 0.4787
Epoch: 16, Loss: 0.4715
Epoch: 17, Loss: 0.4662
Epoch: 18, Loss: 0.4599
Epoch: 19, Loss: 0.4558
Epoch: 20, Loss: 0.4529
Epoch: 21, Loss: 0.4458
Epoch: 22, Loss: 0.4443
Epoch: 23, Loss: 0.4405
Epoch: 24, Loss: 0.4375
Epoch: 25, Loss: 0.4350
Epoch: 26, Loss: 0.4345
Epoch: 27, Loss: 0.4339
Epoch: 28, Loss: 0.4329
Epoch: 29, Loss: 0.4263
Epoch: 30, Loss: 0.4259
Epoch: 31, Loss: 0.4242
Epoch: 32, Loss: 0.4211
Epoch: 33, Loss: 0.4228
Epoch: 34, Loss: 0.4212
Epoch: 35, Loss: 0.4359
Epoch: 36, Loss: 0.4219
Epoch: 37, Loss: 0.4152
Epoch: 38, Loss: 0.4122
Epoch: 39, Loss: 0.4139
Epoch: 40, Loss: 0.4149
Epoch: 41, Loss: 0.4144
Epoch: 42, Loss: 0.4133
E

In [99]:
# prediction

def prediction(model, vocab, text):

  # tokenize
  tokenized_text = word_tokenize(text.lower())

  # text -> numerical indices
  numerical_text = text_to_indices(tokenized_text, vocab)

  # padding
  
  padded_text = torch.tensor([0] * (61 - len(numerical_text)) + numerical_text, dtype=torch.long).unsqueeze(0)
  padded_text = padded_text.to(device)  
  # send to model
  output = model(padded_text)

  # predicted index
  value, index = torch.max(output, dim=1)

  # merge with text
  return text + " " + list(vocab.keys())[index]

In [105]:
prediction(model, vocab, "lets go to the")

'lets go to the kitchen'

In [106]:
import torch

torch.save(model.state_dict(), "lstm_next_word.pth")

In [107]:
import pickle

with open("vocab.pkl", "wb") as f:
    pickle.dump(vocab, f)

In [108]:
idx_to_word = {idx: word for word, idx in vocab.items()}

with open("idx_to_word.pkl", "wb") as f:
    pickle.dump(idx_to_word, f)

In [110]:
with open("len_list.pkl", "wb") as f:
    pickle.dump(len_list, f)